# Fine-tuning Qwen3.8-27B · Kendrick Filbert

Target Advanced: dua eksperimen QLoRA 800 steps, validation dan publikasi merged_16bit. **Draf CPU: training belum dilakukan.**

Jika memakai GPU baru, jalankan `pip install -r requirements-training.txt` dan
`pip install -e . --no-deps`, lalu `pip check` dan restart kernel. Rincian tersedia di
`docs/RUNBOOK_GPU.md`. Ambil token melalui environment/Colab Secrets, jangan tempel di sel.
Untuk mengaktifkan training, set `LEGAL_RAG_TRAIN=1` **sebelum Run All**.
Output CPU yang melewati sel GPU tidak memenuhi persyaratan eksekusi training submission.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("LEGAL_RAG_ROOT", Path.cwd())).resolve()
assert (ROOT / "src/legal_rag").is_dir(), "Set LEGAL_RAG_ROOT to the extracted project folder"
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
from legal_rag.config import settings, read_json, write_json
cfg = settings()
RUN_TRAINING = os.getenv("LEGAL_RAG_TRAIN") == "1"
if RUN_TRAINING:
    import unsloth  # Must precede transformers / sentence_transformers in this kernel.
print("Nama: Kendrick Filbert")
print("Mode:", "GPU training enabled" if RUN_TRAINING else "CPU verification; GPU training NOT executed")
def cli(*args):
    subprocess.run([sys.executable, "-m", "legal_rag.cli", *args], cwd=ROOT, check=True)


In [ ]:
from legal_rag.training import preflight
info = preflight(require_gpu=False)
print(json.dumps(info, indent=2))
print(json.dumps({k:cfg[k] for k in ["base_model", "base_revision", "dataset", "dataset_revision", "experiments"]}, indent=2))

## Dataset resmi, split, mapping native ChatML
Dataset hanya dari tautan tugas. Deduplikasi prompt sebelum pembagian train/validation; seed 3407. Kode mapping berada di `src/legal_rag/data.py`. Cache, bila tersedia, berasal dari eksekusi Dataset.map yang sama.

In [ ]:
from datasets import load_from_disk, DatasetDict
if (ROOT / "data/cache/train").exists():
    ds = DatasetDict({"train": load_from_disk(str(ROOT / "data/cache/train")), "test": load_from_disk(str(ROOT / "data/cache/test"))})
    report = read_json(ROOT / "artifacts/reports/dataset.json")
    print("Loaded verified Dataset.map cache")
else:
    from transformers import PreTrainedTokenizerFast
    from legal_rag.data import prepare_dataset
    tokenizer = PreTrainedTokenizerFast.from_pretrained(cfg["base_model"], revision=cfg["base_revision"])
    ds, report = prepare_dataset(tokenizer)
print(json.dumps(report, indent=2, ensure_ascii=False))
print(ds["train"][0]["text"])
assert "<|im_start|>user" in ds["train"][0]["text"]
assert report["prompt_leakage"] == 0

## NF4 + double quantization + seluruh FFN LoRA
`load_qlora()` mengaktifkan 4-bit NF4 dan `bnb_4bit_use_double_quant=True`, kemudian memverifikasi konfigurasi hidup. Target FFN: gate_proj, up_proj, down_proj. Model Qwen ini memakai hybrid attention. Dokumentasi Unsloth mencatat risiko kuantisasi keluarga Qwen3.5; uji kompatibilitas diperlukan. Tidak ada penggantian model.

In [ ]:
import inspect
from legal_rag.training import load_qlora
print(inspect.getsource(load_qlora))

## Eksperimen A: LR 1e-4, rank 16, alpha 32, 800 optimizer steps
Mulai dari base model resmi. Eval/save setiap 100, logging setiap 10; tidak memakai early stopping sebelum 800 steps.

In [ ]:
if RUN_TRAINING:
    cli("train", "sft_a")
else:
    print("NOT RUN: eksperimen A membutuhkan GPU CUDA. Tidak ada hasil loss yang diklaim.")

## Eksperimen B: LR 5e-5, rank 32, alpha 64, 800 optimizer steps
Proses Python baru memuat base revision yang sama, bukan melanjutkan adapter A.

In [ ]:
if RUN_TRAINING:
    cli("train", "sft_b")
else:
    print("NOT RUN: eksperimen B membutuhkan GPU CUDA.")

In [ ]:
if all((ROOT / f"artifacts/reports/{x}.json").exists() for x in ["sft_a", "sft_b"]):
    from legal_rag.training import compare_experiments
    from legal_rag.evaluation import plot_training
    from IPython.display import display, Image
    print(compare_experiments())
    display(Image(filename=str(plot_training())))
else:
    print("Analisis kurva ditunda: kedua eksperimen belum selesai. Tidak membuat kurva sintetis.")

### Analisis yang diisi dari hasil nyata
Bandingkan train/validation loss, selisih keduanya, checkpoint terbaik, dan indikasi overfitting. Loss lebih rendah pada dataset generik tidak otomatis berarti jawaban legal lebih akurat. Pilihan otomatis memakai validation loss; tuliskan interpretasi berdasarkan kurva sebelum pengumpulan.

## Publikasi hasil SFT
Set `HF_TOKEN` dan `HF_SFT_REPO`. API yang benar: `model.push_to_hub_merged(..., save_method='merged_16bit')`. Verifikasi anonim dan merged weights dilakukan sebelum URL ditulis.

In [ ]:
if RUN_TRAINING:
    cli("publish", "sft")
else:
    print("NOT PUBLISHED: belum ada model SFT milik mahasiswa yang selesai dilatih.")
print((ROOT / "link_huggingface.txt").read_text(encoding="utf-8"))